<a href="https://colab.research.google.com/github/fallgeklaertsaas-dev/DLFI_erste-Ubg/blob/main/01_PyTorch_Linear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Übung 01 - Einführung in PyTorch

In dieser Übung lernen wir erste Schritte mit PyTorch.

PyTorch bietet sehr einfache Möglichkeiten, um Deep Learning Modelle zu erstellen und zu trainieren.  
Um die zugrundeliegenden Konzepte besser zu verstehen, beginnen wir aber mit einem low-level Ansatz.
Das heißt, wir lernen, die auszuführenden Operationen selbst zu definieren.  

In späteren Übungen werden wir, wie in der Praxis üblich, die komfortablen Funktionen nuzten.


### Tensoren in PyTorch

Tensoren sind die Basis-Datenstruktur, mit der PyTorch arbeitet.  
Im Prinzip sind Tensoren nur n-dimensionale Matrizen.  
So ist ein 1-dimensionaler Tensor ein Vektor und ein 2-dimensionaler Tensor eine Matrix.  
Alle Variablen, die wir mit torch-Funktionen erstellen, sind automatisch torch-Tensoren.  
Variablen als torch-Tensoren zu verwenden ist wichtig, damit PyTorch einen Computation Graph erstellen kann.  
Die Verwendung ist dabei einfach und viele Funktionen aus `numpy` sind übertragbar.

In [ ]:
import numpy as np
import torch

In [ ]:
lin_space_numpy = np.linspace(0,10,11)
print(type(lin_space_numpy), lin_space_numpy)

lin_space_torch = torch.linspace(0,10,11)
print(type(lin_space_torch), lin_space_torch)

## Datensatz

Zunächst benötigen wir einen Datensatz, den wir für das Modell-Training verwenden können.  
Wir nutzen dafür den MNIST Datensatz, eine Sammlung von handgeschriebenen Ziffern als 28x28 px Graustufen-Bilder.  
Als sehr einfacher Datensatz wird MNIST oft für Demonstrations-Zwecke verwendet.

### Download
MNIST (sowie andere typische Datensätze) lässt sich leicht über `torchvision` herunterladen.  

In [ ]:
import torch
import torchvision

In [ ]:
# Der Datensatz wird geladen und ist geteilt in Trainings- und Test-Daten, sowie die jeweiligen Annotationen
trainset = torchvision.datasets.MNIST(root='./data/', train=True, download=True)
testset = torchvision.datasets.MNIST(root='./data/', train=False, download=True)

In [ ]:
# die Datensätze enthalten Datenpunkte (Bilder) und deren Klassen, in `.data` und `.targets`

print("Dimensionen der Trainings-Bilder:", trainset.data.shape)
print("Dimensionen der Trainings-Label: ", trainset.targets.shape)

print("Dimensionen der Test-Bilder:", testset.data.shape)
print("Dimensionen der Test-Label: ", testset.targets.shape)

Wie sieht der Datensatz aus?  
Dazu schauen wir die Dimension der Daten und Label an und plotten Beispiele.  

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
example_id = 5
print("Bild mit Index %d der Trainings-Daten:" % (example_id))
plt.imshow(trainset.data[example_id], cmap="Greys_r")
plt.title("Label: %d" % (trainset.targets[example_id]))
plt.xticks([])
plt.yticks([])
plt.show()

### Daten für das Training vorbereiten

Die MNIST-Bilder sind 2-dimensional, aber für die Modelle in dieser Übung betrachten wir die Daten als 1-dimensional.  
Darum ändern wir die Dimensionen der 28$\times$28 Bilder zu einer Dimension mit 784 Werten.  
Außerdem sind die Pixelwerte als Integer definiert (0-255), wir benötigen aber Fließkommazahlen (float) für das Modell.  
Bei der Gelegenheit skalieren wir die Pixelwerte einfach im Bereich `[0,1]`.

In [ ]:
trainset.flat_data = trainset.data.reshape([-1, 784])
trainset.flat_data = trainset.flat_data / trainset.flat_data.max()
testset.flat_data = testset.data.reshape([-1, 784])
testset.flat_data = testset.flat_data / testset.flat_data.max()

##Lineares Modell

Jetzt werden wir unser erstes Modell in DLFI trainieren und starten mit einem einfachen linearen Modell.  

Wären die Eingaben $x$ und Ausgaben $y$ skalare Werte, würden wir die lineare Gleichung  
$y = mx + n$  
optimieren, mit $m$ und $n$ als skalare Parameter für Anstieg und Ordinatenabschnitt.

Unsere Eingaben sind allerdings Vektoren der Länge 784 und unsere Ausgaben Vektoren der Länge 10 (10 Ausgabeneuronen, weil 10 mögliche Klassen).
Das lineare Modell ist - in Matrixschreibweise - entsprechend  
$y = Wx+b$  
mit den Variablen:  
$W$ ... Gewichtsmatrix, also 784$\times$10 Werte  
$b$ ... Bias für jede Ausgabe, also 10 Werte   

### Variablen und Trainingsparameter definieren

Zunächst definieren wir die Variablen $W$ und $b$.  
Die Variablen müssen explizit als PyTorch-Variablen definiert werden, da wir sie beim Training optimieren wollen.  
Durch den `requires_grad=True` Parameter signalisieren wir, dass später Gradienten für diese Variablen ausgegeben werden sollen.  
Außerdem legen wir Hyperparameter _learning rate_ und _Anzahl Schritte_ für unser Training fest.  

In [ ]:
W = torch.zeros([784, 10], dtype=torch.float, requires_grad=True)
b = torch.zeros(10, dtype=torch.float, requires_grad=True)

learning_rate = 0.1
train_steps = 1000

### Trainings-Schleife

Nun definieren wir das Training.  
In jedem Trainingsschritt wird ein neuer Batch aus den Daten genommen.  
Dieser wird entsprechend dem aktuellen linearen Modell $y = Wx+b$ verarbeitet.  
Zu Beginn sind die Ausgaben wahrscheinlich falsch, es entsteht also ein Fehler.  
Den berechnen wir als die Kreuzentropie zwischen den vorhergesagten und den wahren Ausgaben.  
Um das Modell zu verbessern, berechnen wir die Gradienten des Fehlers bezüglich der zu lernenden Variablen, also $W$ und $b$.  
Dann optimieren wir die Variablen, indem wir Gradientenabstieg mit der Lernrate anwenden.  
Nach einigen Schritten sollten wir unser lineares Modell damit deutlich verbessert haben.

_Anmerkung:_  
_Die Kreuzentropie wird eigentlich zwischen den vorhergesagten und gewünschten Wahrscheinlichkeiten berechnet. Die Implementierung mit_ `torch.nn.functional.cross_entropy` _nimmt allerdings die unnormalisierten Wahrscheinlichkeiten an (logits)._  
_Auf den softmax in der Ausgabeschicht können wir demenstprechend bei der Implementierung verzichten._
_Andere Loss-Implementierungen, wie z.B._ `torch.nn.functional.binary_cross_entropy` _gehen hingegen von Wahrscheinlichkeiten aus und benötigen sigmoid bzw. softmax im Output._

In [ ]:
# kleine vereinfachte Hilfsfunktion, um Minibatches aus den Daten zu ziehen
# sobald alle Daten durchgegangen sind, wird die Reihenfolge permutiert und neu von vorne begonnen

def get_next_batch(dataset, use_flat=True, batch_size=32, shuffle=True):
    try:
      dataset.last_batch_start_idx
    except:
      dataset.last_batch_start_idx = 0
    from_idx = dataset.last_batch_start_idx
    to_idx = from_idx+batch_size

    if use_flat:
      image_batch = dataset.flat_data[from_idx:to_idx]
    else:
      image_batch = dataset.data[from_idx:to_idx]
    label_batch = dataset.targets[from_idx:to_idx]

    dataset.last_batch_start_idx = to_idx
    if to_idx > dataset.data.shape[0]:
      dataset.last_batch_start_idx = 0
      if shuffle:
        shuffle_order = torch.randperm(dataset.data.shape[0])
        dataset.data = dataset.data[shuffle_order]
        dataset.flat_data = dataset.flat_data[shuffle_order]
        dataset.targets = dataset.targets[shuffle_order]

    return image_batch, label_batch

In [ ]:
loss_fn = torch.nn.functional.cross_entropy

for step in range(train_steps):
    img_batch, lbl_batch = get_next_batch(trainset)

    # forward pass definieren
    # in unserem Fall ist es nur ein lineares Modell
    logits = torch.matmul(img_batch, W) + b

    # loss berechnen
    batch_loss = loss_fn(logits,
                         lbl_batch,
                         reduction='mean' # loss wird für jedes Beispiel berechnet und dann gemittelt
                         )

    # Um zu wissen, wie unsere Variablen optimiert werden müssen,
    # brauchen wir einen backward pass, in dem die Gradienten bzgl. der Variablen
    # berechnet werden.
    # Glücklicherweise bietet torch dafür die autograd Funktionalität an.
    # Wir geben an, wovon die Gradienten berechnet werden (typischerweise der loss)
    # und für alle `requires_grad=True` Variablen werden die Gradienten berechnet.
    batch_loss.backward()

    with torch.no_grad():  # durch die Updates der Gewichte muss kein gradient berechnet werden
        # Gradienten tauchen nach `backward()` als `.grad` an den Variablen selbst auf
        # Diese können wir für den Gradientenabstieg verwenden
        W -= learning_rate * W.grad
        b -= learning_rate * b.grad

        # Gradienten zurücksetzen
        W.grad = None
        b.grad = None


        if not step % 100:
            preds = torch.argmax(logits, dim=1)
            acc = torch.mean((preds == lbl_batch).double())
            print("Batch-Loss: {:.3f} Batch-Accuracy: {:.3f}".format(batch_loss.numpy(), acc))

Die Genauigkeit auf den Trainingsdaten sieht jetzt schon gut aus.  
Aber es ist auch wichtig, dass das Modell generalisiert.  
Darum evaluieren wir das Modell zusätzlich auf den Test-Daten.

## ❗Aufgabe 01-01 Test-Accuracy
Ermittle die Genauigkeit auf den Test-Daten!

Für das kleine Modell und MNIST kann der gesamte Testdatensatz auf einmal evaluiert werden (sogar der gesamte Trainings-Datensatz).    
Für gewöhnlich muss aber auch der Testdatensatz in Batches verarbeitet werden, um ihn zu evaluieren.  
Schreibe den Code am besten so, dass Vorhersagegenauigkeiten für mehrere Batches berechnet und aggregiert werden!

Zusätzlich wäre auch eine Konfusionsmatrix zur Auswertung interessant.  
Welche Klassen wurden besonders gut/schlecht klassifiziert?


Wir sollten sehen, dass MNIST recht einfach zu klassifizieren ist.  
Sogar ein einfaches lineares Modell erreicht Vorhersage-Genauigkeit von ca. 90%.

## ❗Aufgabe 01-02 Lernkurven visualisieren

Stelle die Lernkurve des Modells als Plot dar!  
Ein Plot stellt den Verlauf des Losses dar, ein weiterer den Verlauf der Vorhersagegenauigkeit.  
Beide Plots zeigen jeweils vergleichend Training und Test.

Sind die Ergebnisse des Plots sinnvoll?   
Erkennst du, wie das Modell durch das Training besser wird?   

Hinweise:
*   Im Moment wird nur am Ende des Trainings die Test-Accuracy berechnet.   
  Für einen Plot muss dieser Wert häufiger ermittelt werden.
*   `matplotlib` bietet viele Möglichkeiten, Plots zu erstellen (`import matplotlib.pyplot as plt`)  
   `plotly` bietet auch interaktive Elemente  
  Andere Bibliotheken sind natürlich auch möglich.


## ❗Aufgabe 01-03 Gewichte visualisieren

Welche Muster werden vom linearen Modell gelernt?  
  Eine simple Möglichkeit ist, die Gewichte zu visualisieren.  
  1. Man nehme alle eingehenden Gewichte für ein Ausgabenneuron,
  2. dann ändert man die Dimension von 784 zu 28$\times$28
  3. und plottet die Matrix als Heatmap-Bild (z.B. `plt.imshow()` mit einer `cmap`)

## 💡 Bonus-Aufgabe 01-04

Wie wirken sich die Parameter Lernrate und Anzahl der Trainingsschritte auf den Verlauf der Lernkurve aus?  
Was ist deine Erwartung und bestätigt sich diese durch Experimente mit dem linearen Modell?
